# LUMI 1단계: 단일 정지 이미지 내 광과민성(PSE) 위험 여부 판정 알고리즘 (`image_discriminator_1.ipynb`)

본 노트북은 단일 정지 이미지(1 프레임)를 입력받아 시간축 조건을 제외한 **4가지 공간·색상·휘도 위험 요소**를 **2가지 화면 면적 임계치(25%, 40%)**로 각각 판정하는 **총 8개의 독립 코드 셀(구성안 B)**로 구성되어 있습니다.

### 적용된 정량 기준 근거
1. **픽셀 RGB $\rightarrow$ 물리 휘도($L$, $\text{cd/m}^2$) 변환 (방식 A — ITU-R BT.1702-3 / WCAG 2.2)**
   - 8-bit sRGB 채널 $C_{8\text{bit}} \in [0, 255]$를 $C_s = C_{8\text{bit}} / 255.0$로 정규화한 후 IEC 61966-2-1 감마 복원을 통해 선형 RGB $C_{\text{lin}} \in [0, 1]$ 계산:
     $$C_{\text{lin}} = \begin{cases} \frac{C_s}{12.92} & \text{if } C_s \le 0.04045 \\ \left(\frac{C_s + 0.055}{1.055}\right)^{2.4} & \text{if } C_s > 0.04045 \end{cases}$$
   - 상대 휘도 $Y = 0.2126 R_{\text{lin}} + 0.7152 G_{\text{lin}} + 0.0722 B_{\text{lin}}$ 및 SDR 피크 화이트 $200\text{ cd/m}^2$ 기준 물리 휘도 $L = 200.0 \times Y\ (\text{cd/m}^2)$ 산출.
2. **명암 대비(휘도 차) 위험 기준 (ITU-R BT.1702-3 Guideline 1 & Attachment 1)**
   - 어두운 영역 휘도 $L_{\text{LOW}} < 160\text{ cd/m}^2$일 때: $\Delta L = L_{\text{HIGH}} - L_{\text{LOW}} \ge 20\text{ cd/m}^2$
   - 어두운 영역 휘도 $L_{\text{LOW}} \ge 160\text{ cd/m}^2$일 때: Michelson 대비 $C_M = \frac{L_{\text{HIGH}} - L_{\text{LOW}}}{L_{\text{HIGH}} + L_{\text{LOW}}} > \frac{1}{17}$
3. **4가지 위험 요소 및 면적 임계치 (25% 초과 / 40% 초과)**
   - **유형 1 (셀 1~2) — 국소 공간 패턴 통합**: 직선 줄무늬(다방향 투영), 원형(동심원)·방사형(극좌표 변환), 반복 무늬(2D FFT 및 격자 프로파일)에서 명암 대비 기준을 충족하는 **명암 쌍(light-dark pairs)이 5개 이상($\ge 5$)** 존재하고, 패턴 영역 면적이 화면의 **25% 초과(셀 1)** 또는 **40% 초과(셀 2)**일 때 위험.
   - **유형 2 (셀 3~4) — 국소 고대비 영역**: 국소 영역 내 $L_{\text{HIGH}}, L_{\text{LOW}}$가 ITU-R BT.1702-3 명암 대비 기준을 초과하는 영역 면적이 화면의 **25% 초과(셀 3)** 또는 **40% 초과(셀 4)**일 때 위험.
   - **유형 3 (셀 5~6) — 국소 고휘도 영역**: Optum 특허(US11769465B1) 기준에 따라 프레임 평균 기준 휘도($L_{\text{base}}$) 대비 일정 비율(**$+30\%$ 이상**, 최소 유의 휘도 $20\text{ cd/m}^2$ 이상) 높은 고휘도 영역 면적이 화면의 **25% 초과(셀 5)** 또는 **40% 초과(셀 6)**일 때 위험.
   - **유형 4 (셀 7~8) — 포화 적색 영역**: WCAG 2.2 / ISO 9241-391 기준에 따라 선형 RGB 적색 비율 $\frac{R_{\text{lin}}}{R_{\text{lin}} + G_{\text{lin}} + B_{\text{lin}}} \ge 0.8$인 영역 면적이 **25% 초과(셀 7)** 또는 **40% 초과(셀 8)**이면서, 동시에 프레임 내 타 영역과의 **CIE 1976 UCS 색도 거리 $\Delta u'v' > 0.2$**를 만족할 때 위험.
4. **판정 근거 확률 3종 비교 테이블 출력**
   - **방식 1 (임계치 달성도 기반 확률, `%`)**: 면적 임계치 달성 비율과 세부 조건(대비·줄무늬 쌍·적색비·색도 거리) 달성도를 결합한 정규화 점수 $\eta$에 대해 $\frac{\eta^2}{1 + \eta^2} \times 100\%$ 매핑 (임계치 정확히 도달 시 $\eta = 1 \implies 50.0\%$, 초과 시 $>50\%$).
   - **방식 2 (위험 조건 충족 면적 비율, `%`)**: 해당 위험 조건을 만족하는 픽셀이 전체 화면에서 차지하는 실제 면적 비율($0\sim 100\%$). (단, 부가 필수 조건 미달 시 유효 위험 면적 $0\%$ 처리 후 원시 면적은 참고 가능하도록 결합)
   - **방식 3 (시피질 자극 $Q$ 기반 환자 위험 확률, `%`)**: Jordan & Vanderheiden (2024, `3694790.pdf` Appendix A) 및 Binnie et al. (2002)의 대뇌 시각피질 자극 비율 $Q = 100(1 - e^{-0.0574\theta})$로부터 산출된 PSE 환자 발작 위험 비율($0.021Q - 0.1827$, $Q \in [8.75, 56.37]$ 구간에서 $[0, 1]$ 선형 연속 매핑 후 $\%$ 환산; 원 논문 부록의 오타인 $0.21Q$를 경계 연속 조건 $Q=8.75 \implies 0,\ Q=56.37 \implies 1$에 맞춰 $0.021Q$로 보정).

### [코드 셀 1] 국소 공간 패턴 통합(직선 줄무늬 · 원형/방사형 · 반복 무늬) 판정 알고리즘 — 면적 임계치 25%

In [ ]:
import os
import math
import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from IPython.display import display

# 분석할 단일 이미지 경로 (이미지 파일 또는 영상 파일 입력 시 중간 프레임 자동 추출)
IMAGE_PATH = r"이미지 경로"

@torch.no_grad()
def load_single_image_rgb(path: str, max_side: int = 1080) -> torch.Tensor:
    """단일 정지 이미지를 로드하여 GPU/CPU 텐서 (H, W, 3), float32 [0, 255] 형태로 반환합니다."""
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    if not os.path.exists(path):
        raise FileNotFoundError(f"입력 파일을 찾을 수 없습니다: {path}")
    ext = os.path.splitext(path)[1].lower()
    if ext in [".mp4", ".avi", ".mov", ".mkv", ".webm"]:
        cap = cv2.VideoCapture(path)
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        if total_frames > 0:
            cap.set(cv2.CAP_PROP_POS_FRAMES, total_frames // 2)
        ret, frame_bgr = cap.read()
        cap.release()
        if not ret or frame_bgr is None:
            raise ValueError(f"영상에서 프레임을 읽을 수 없습니다: {path}")
        img_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    else:
        buf = np.fromfile(path, dtype=np.uint8)
        frame_bgr = cv2.imdecode(buf, cv2.IMREAD_COLOR)
        if frame_bgr is None:
            raise ValueError(f"이미지를 읽을 수 없습니다: {path}")
        img_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)

    h, w = img_rgb.shape[:2]
    if max(h, w) > max_side:
        scale = max_side / float(max(h, w))
        img_rgb = cv2.resize(img_rgb, (int(w * scale), int(h * scale)), interpolation=cv2.INTER_AREA)
    return torch.from_numpy(img_rgb).to(device=device, dtype=torch.float32)

@torch.no_grad()
def srgb_to_linear_and_luminance(rgb_255: torch.Tensor):
    """방식 A (ITU-R BT.1702-3 / WCAG 2.2): sRGB [0,255] -> 선형 RGB [0,1] 및 물리 휘도 L [0, 200] cd/m^2"""
    c_s = torch.clamp(rgb_255 / 255.0, 0.0, 1.0)
    c_lin = torch.where(
        c_s <= 0.04045,
        c_s / 12.92,
        torch.pow((c_s + 0.055) / 1.055, 2.4)
    )
    y_rel = 0.2126 * c_lin[..., 0] + 0.7152 * c_lin[..., 1] + 0.0722 * c_lin[..., 2]
    lum_cd = 200.0 * y_rel
    return c_lin, y_rel, lum_cd

def compute_cortical_risk_prob(area_fraction: float, quality_gate: float = 1.0, theta_full_deg: float = 12.53) -> float:
    """방식 3 (3694790.pdf Section 5.4 / Binnie et al.): 시피질 자극 비율 Q = 100(1 - exp(-0.0574*theta)) 기반 환자 위험 확률(%)"""
    eff_area = max(0.0, min(1.0, float(area_fraction) * float(quality_gate)))
    if eff_area <= 0.0:
        return 0.0
    theta = theta_full_deg * math.sqrt(eff_area)
    q_val = 100.0 * (1.0 - math.exp(-0.0574 * theta))
    if q_val > 56.37:
        prop = 1.0
    elif q_val > 8.75:
        # 3694790.pdf Appendix A의 경계값(Q=8.75 -> 0, Q=56.37 -> 1) 연속성 기준 기울기 0.021 적용
        prop = 0.021 * q_val - 0.1827
    else:
        prop = 0.0
    return float(np.clip(prop, 0.0, 1.0) * 100.0)

def count_opposing_stripe_pairs_1d(profile: np.ndarray) -> int:
    """1차원 휘도 프로파일에서 ITU-R BT.1702-3 명암 대비 기준(>=20 cd/m^2 또는 Michelson > 1/17)을 만족하는 밝고 어두운 줄무늬 쌍(light-dark pairs) 개수를 계산합니다."""
    if profile.size < 4:
        return 0
    extrema_vals = [float(profile[0])]
    direction = 0
    for v in profile[1:]:
        v = float(v)
        diff = v - extrema_vals[-1]
        if direction >= 0 and diff > 0:
            if direction == 0:
                extrema_vals.append(v)
            else:
                extrema_vals[-1] = v
            direction = 1
        elif direction <= 0 and diff < 0:
            if direction == 0:
                extrema_vals.append(v)
            else:
                extrema_vals[-1] = v
            direction = -1
        elif direction == 1 and (extrema_vals[-1] - v) >= 5.0:
            extrema_vals.append(v)
            direction = -1
        elif direction == -1 and (v - extrema_vals[-1]) >= 5.0:
            extrema_vals.append(v)
            direction = 1

    valid_transitions = 0
    for i in range(len(extrema_vals) - 1):
        l_high = max(extrema_vals[i], extrema_vals[i + 1])
        l_low = min(extrema_vals[i], extrema_vals[i + 1])
        cond_low = (l_low < 160.0) and ((l_high - l_low) >= 20.0)
        cond_high = (l_low >= 160.0) and (((l_high - l_low) / max(l_high + l_low, 1e-6)) > (1.0 / 17.0))
        if cond_low or cond_high:
            valid_transitions += 1
    return valid_transitions // 2

@torch.no_grad()
def evaluate_spatial_pattern(image_path: str, area_threshold: float = 0.25) -> pd.DataFrame:
    rgb = load_single_image_rgb(image_path)
    _, _, lum = srgb_to_linear_and_luminance(rgb)
    h, w = lum.shape

    # 1. 국소 명암 대비(ITU-R BT.1702-3) 마스크 및 심각도 비율 계산
    k_size = max(7, int(min(h, w) * 0.03) | 1)
    pad = k_size // 2
    lum_4d = lum.unsqueeze(0).unsqueeze(0)
    l_high = F.max_pool2d(lum_4d, kernel_size=k_size, stride=1, padding=pad).squeeze()
    l_low = -F.max_pool2d(-lum_4d, kernel_size=k_size, stride=1, padding=pad).squeeze()

    delta_l = l_high - l_low
    michelson = delta_l / torch.clamp(l_high + l_low, min=1e-6)
    contrast_ratio = torch.where(l_low < 160.0, delta_l / 20.0, michelson / (1.0 / 17.0))
    contrast_mask = torch.where(l_low < 160.0, delta_l >= 20.0, michelson > (1.0 / 17.0))

    # 국소 공간 패턴 영역(고주파 에지/주기성 밀집 영역) 스무딩으로 패턴 점유 영역 산출
    env_k = max(15, int(min(h, w) * 0.06) | 1)
    pattern_density = F.avg_pool2d(contrast_mask.float().unsqueeze(0).unsqueeze(0), kernel_size=env_k, stride=1, padding=env_k // 2).squeeze()
    pattern_region_mask = pattern_density >= 0.15
    pattern_area_fraction = float(pattern_region_mask.float().mean().item())

    # 2. 기하 구조별 줄무늬 쌍(light-dark pairs) 카운팅 (직선 다방향 / 동심원·방사형 / 2D 반복 격자)
    lum_np = lum.detach().cpu().numpy()
    max_pairs = 0

    # (a) 직선 줄무늬: 0~165도 다방향 단면 스캔
    cy, cx = h / 2.0, w / 2.0
    radius = int(min(h, w) * 0.45)
    for deg in range(0, 180, 15):
        rad = math.radians(deg)
        for offset in np.linspace(-radius * 0.5, radius * 0.5, 5):
            t_vals = np.linspace(-radius, radius, 256)
            xs = np.clip((cx + t_vals * math.cos(rad) - offset * math.sin(rad)).astype(int), 0, w - 1)
            ys = np.clip((cy + t_vals * math.sin(rad) + offset * math.cos(rad)).astype(int), 0, h - 1)
            pairs = count_opposing_stripe_pairs_1d(lum_np[ys, xs])
            if pairs > max_pairs:
                max_pairs = pairs

    # (b) 원형(동심원) 및 방사형 패턴: 극좌표(r, theta) 프로파일 스캔
    for ang_deg in range(0, 360, 30):
        rad = math.radians(ang_deg)
        r_vals = np.linspace(0, radius, 180)
        xs = np.clip((cx + r_vals * math.cos(rad)).astype(int), 0, w - 1)
        ys = np.clip((cy + r_vals * math.sin(rad)).astype(int), 0, h - 1)
        pairs_concentric = count_opposing_stripe_pairs_1d(lum_np[ys, xs])
        if pairs_concentric > max_pairs:
            max_pairs = pairs_concentric

    for r_frac in [0.25, 0.5, 0.75, 0.9]:
        r_cur = radius * r_frac
        thetas = np.linspace(0, 2.0 * math.pi, 360, endpoint=False)
        xs = np.clip((cx + r_cur * np.cos(thetas)).astype(int), 0, w - 1)
        ys = np.clip((cy + r_cur * np.sin(thetas)).astype(int), 0, h - 1)
        pairs_radial = count_opposing_stripe_pairs_1d(lum_np[ys, xs])
        if pairs_radial > max_pairs:
            max_pairs = pairs_radial

    # 3. 최종 위험/안전 판정 (줄무늬 쌍 >= 5개 이고 패턴 면적 > 임계치)
    is_danger = (max_pairs >= 5) and (pattern_area_fraction > area_threshold)
    verdict = "위험 (DANGER)" if is_danger else "안전 (SAFE)"

    # 4. 판정 근거 확률 3가지 방식 계산
    mean_contrast_factor = float(torch.clamp(contrast_ratio[pattern_region_mask].mean() if pattern_region_mask.any() else contrast_ratio.max(), 0.0, 1.0).item())
    pair_factor = min(1.0, max_pairs / 5.0)
    eta = (pattern_area_fraction / area_threshold) * pair_factor * mean_contrast_factor
    prob_method_1 = (eta ** 2 / (1.0 + eta ** 2)) * 100.0
    prob_method_2 = (pattern_area_fraction * 100.0) if (max_pairs >= 5) else 0.0
    prob_method_3 = compute_cortical_risk_prob(pattern_area_fraction, quality_gate=pair_factor * mean_contrast_factor)

    return pd.DataFrame([{
        "판정 결과": verdict,
        "판정 근거 확률 - 방식 1 (임계치 달성도 기반)": f"{prob_method_1:.2f}%",
        "판정 근거 확률 - 방식 2 (위험 조건 충족 면적 비율)": f"{prob_method_2:.2f}%",
        "판정 근거 확률 - 방식 3 (시피질 자극 Q 기반 환자 위험도)": f"{prob_method_3:.2f}%"
    }], index=["국소 공간 패턴 통합 (면적 기준 25%)"])

df_cell_1 = evaluate_spatial_pattern(IMAGE_PATH, area_threshold=0.25)
display(df_cell_1)

,판정 결과,판정 근거 확률 - 방식 1 (임계치 달성도 기반),판정 근거 확률 - 방식 2 (위험 조건 충족 면적 비율),판정 근거 확률 - 방식 3 (시피질 자극 Q 기반 환자 위험도)
국소 공간 패턴 통합 (면적 기준 25%),위험 (DANGER),94.12%,99.98%,89.43%


### [코드 셀 2] 국소 공간 패턴 통합(직선 줄무늬 · 원형/방사형 · 반복 무늬) 판정 알고리즘 — 면적 임계치 40%

In [4]:
# 분석할 단일 이미지 경로 (코드 셀 1과 동일하거나 개별 지정 가능)

@torch.no_grad()
def evaluate_spatial_pattern_40(image_path: str) -> pd.DataFrame:
    df_res = evaluate_spatial_pattern(image_path, area_threshold=0.40)
    df_res.index = ["국소 공간 패턴 통합 (면적 기준 40%)"]
    return df_res

df_cell_2 = evaluate_spatial_pattern_40(IMAGE_PATH)
display(df_cell_2)

,판정 결과,판정 근거 확률 - 방식 1 (임계치 달성도 기반),판정 근거 확률 - 방식 2 (위험 조건 충족 면적 비율),판정 근거 확률 - 방식 3 (시피질 자극 Q 기반 환자 위험도)
국소 공간 패턴 통합 (면적 기준 40%),위험 (DANGER),86.20%,99.98%,89.43%


### [코드 셀 3] 국소 고대비 영역 판정 알고리즘 — 면적 임계치 25%

In [5]:
@torch.no_grad()
def evaluate_high_contrast(image_path: str, area_threshold: float = 0.25) -> pd.DataFrame:
    """
    ITU-R BT.1702-3 명암 대비 기준 적용:
    - 어두운 영역 L_LOW < 160 cd/m^2 일 때: L_HIGH - L_LOW >= 20 cd/m^2
    - 어두운 영역 L_LOW >= 160 cd/m^2 일 때: Michelson 대비 (L_HIGH - L_LOW)/(L_HIGH + L_LOW) > 1/17
    """
    rgb = load_single_image_rgb(image_path)
    _, _, lum = srgb_to_linear_and_luminance(rgb)
    h, w = lum.shape

    # 국소 영역 내 최대/최소 휘도 산출
    k_size = max(9, int(min(h, w) * 0.04) | 1)
    pad = k_size // 2
    lum_4d = lum.unsqueeze(0).unsqueeze(0)
    l_high = F.max_pool2d(lum_4d, kernel_size=k_size, stride=1, padding=pad).squeeze()
    l_low = -F.max_pool2d(-lum_4d, kernel_size=k_size, stride=1, padding=pad).squeeze()

    delta_l = l_high - l_low
    michelson = delta_l / torch.clamp(l_high + l_low, min=1e-6)

    high_contrast_mask = torch.where(
        l_low < 160.0,
        delta_l >= 20.0,
        michelson > (1.0 / 17.0)
    )
    severity_ratio = torch.where(
        l_low < 160.0,
        delta_l / 20.0,
        michelson / (1.0 / 17.0)
    )

    contrast_area_fraction = float(high_contrast_mask.float().mean().item())
    is_danger = contrast_area_fraction > area_threshold
    verdict = "위험 (DANGER)" if is_danger else "안전 (SAFE)"

    # 방식 1, 2, 3 확률 산출
    intensity_gate = float(torch.clamp(severity_ratio[high_contrast_mask].mean() if high_contrast_mask.any() else severity_ratio.max(), 0.0, 1.0).item())
    eta = (contrast_area_fraction / area_threshold) * intensity_gate
    prob_method_1 = (eta ** 2 / (1.0 + eta ** 2)) * 100.0
    prob_method_2 = contrast_area_fraction * 100.0
    prob_method_3 = compute_cortical_risk_prob(contrast_area_fraction, quality_gate=intensity_gate)

    label_pct = int(round(area_threshold * 100))
    return pd.DataFrame([{
        "판정 결과": verdict,
        "판정 근거 확률 - 방식 1 (임계치 달성도 기반)": f"{prob_method_1:.2f}%",
        "판정 근거 확률 - 방식 2 (위험 조건 충족 면적 비율)": f"{prob_method_2:.2f}%",
        "판정 근거 확률 - 방식 3 (시피질 자극 Q 기반 환자 위험도)": f"{prob_method_3:.2f}%"
    }], index=[f"국소 고대비 영역 (면적 기준 {label_pct}%)"])

df_cell_3 = evaluate_high_contrast(IMAGE_PATH, area_threshold=0.25)
display(df_cell_3)

,판정 결과,판정 근거 확률 - 방식 1 (임계치 달성도 기반),판정 근거 확률 - 방식 2 (위험 조건 충족 면적 비율),판정 근거 확률 - 방식 3 (시피질 자극 Q 기반 환자 위험도)
국소 고대비 영역 (면적 기준 25%),위험 (DANGER),93.84%,97.59%,88.54%


### [코드 셀 4] 국소 고대비 영역 판정 알고리즘 — 면적 임계치 40%

In [6]:

df_cell_4 = evaluate_high_contrast(IMAGE_PATH, area_threshold=0.40)
display(df_cell_4)

,판정 결과,판정 근거 확률 - 방식 1 (임계치 달성도 기반),판정 근거 확률 - 방식 2 (위험 조건 충족 면적 비율),판정 근거 확률 - 방식 3 (시피질 자극 Q 기반 환자 위험도)
국소 고대비 영역 (면적 기준 40%),위험 (DANGER),85.62%,97.59%,88.54%


### [코드 셀 5] 국소 고휘도 영역 판정 알고리즘 — 면적 임계치 25%

In [7]:

@torch.no_grad()
def evaluate_high_luminance(image_path: str, area_threshold: float = 0.25, rel_exceed_ratio: float = 0.30) -> pd.DataFrame:
    """
    Optum 특허(US11769465B1) 기반 상대 비율 고휘도 판정:
    - 프레임 기준 휘도(평균 휘도 L_base) 대비 +30% 이상(L >= L_base * 1.30) 밝고,
      최소 유의 휘도(20 cd/m^2) 이상인 국소 고휘도 픽셀의 면적 비율로 판정합니다.
    """
    rgb = load_single_image_rgb(image_path)
    _, _, lum = srgb_to_linear_and_luminance(rgb)

    l_base = torch.clamp(lum.mean(), min=1.0)
    target_lum = torch.clamp(l_base * (1.0 + rel_exceed_ratio), min=20.0)

    high_lum_mask = lum >= target_lum
    high_lum_area_fraction = float(high_lum_mask.float().mean().item())

    is_danger = high_lum_area_fraction > area_threshold
    verdict = "위험 (DANGER)" if is_danger else "안전 (SAFE)"

    # 방식 1, 2, 3 확률 산출
    rel_ratio_map = torch.clamp((lum - l_base) / torch.clamp(l_base * rel_exceed_ratio, min=1e-6), min=0.0)
    lum_gate = float(torch.clamp(rel_ratio_map[high_lum_mask].mean() if high_lum_mask.any() else rel_ratio_map.max(), 0.0, 1.0).item())
    eta = (high_lum_area_fraction / area_threshold) * lum_gate
    prob_method_1 = (eta ** 2 / (1.0 + eta ** 2)) * 100.0
    prob_method_2 = high_lum_area_fraction * 100.0
    prob_method_3 = compute_cortical_risk_prob(high_lum_area_fraction, quality_gate=lum_gate)

    label_pct = int(round(area_threshold * 100))
    return pd.DataFrame([{
        "판정 결과": verdict,
        "판정 근거 확률 - 방식 1 (임계치 달성도 기반)": f"{prob_method_1:.2f}%",
        "판정 근거 확률 - 방식 2 (위험 조건 충족 면적 비율)": f"{prob_method_2:.2f}%",
        "판정 근거 확률 - 방식 3 (시피질 자극 Q 기반 환자 위험도)": f"{prob_method_3:.2f}%"
    }], index=[f"국소 고휘도 영역 (면적 기준 {label_pct}%)"])

df_cell_5 = evaluate_high_luminance(IMAGE_PATH, area_threshold=0.25)
display(df_cell_5)

,판정 결과,판정 근거 확률 - 방식 1 (임계치 달성도 기반),판정 근거 확률 - 방식 2 (위험 조건 충족 면적 비율),판정 근거 확률 - 방식 3 (시피질 자극 Q 기반 환자 위험도)
국소 고휘도 영역 (면적 기준 25%),위험 (DANGER),69.66%,37.88%,56.84%


### [코드 셀 6] 국소 고휘도 영역 판정 알고리즘 — 면적 임계치 40%

In [8]:

df_cell_6 = evaluate_high_luminance(IMAGE_PATH, area_threshold=0.40)
display(df_cell_6)

,판정 결과,판정 근거 확률 - 방식 1 (임계치 달성도 기반),판정 근거 확률 - 방식 2 (위험 조건 충족 면적 비율),판정 근거 확률 - 방식 3 (시피질 자극 Q 기반 환자 위험도)
국소 고휘도 영역 (면적 기준 40%),안전 (SAFE),47.28%,37.88%,56.84%


### [코드 셀 7] 포화 적색 영역 판정 알고리즘 — 면적 임계치 25%

In [9]:

@torch.no_grad()
def evaluate_saturated_red(image_path: str, area_threshold: float = 0.25) -> pd.DataFrame:
    """
    WCAG 2.2 / ISO 9241-391 포화 적색 기준:
    1) 선형 RGB 기준 R / (R + G + B) >= 0.8 인 포화 적색 픽셀 면적 비율 > area_threshold (25% 또는 40%)
    2) 프레임 내 비적색(또는 대조 색상) 영역과의 CIE 1976 UCS 색도 거리 Delta u'v' > 0.2
    두 조건을 동시에 만족할 때 위험으로 판정합니다.
    """
    rgb = load_single_image_rgb(image_path)
    c_lin, _, _ = srgb_to_linear_and_luminance(rgb)
    r_lin, g_lin, b_lin = c_lin[..., 0], c_lin[..., 1], c_lin[..., 2]

    # 1. 포화 적색 비율 R / (R + G + B) >= 0.8
    rgb_sum = r_lin + g_lin + b_lin
    red_ratio = torch.where(rgb_sum > 1e-4, r_lin / torch.clamp(rgb_sum, min=1e-6), torch.zeros_like(r_lin))
    sat_red_mask = (red_ratio >= 0.8) & (r_lin > 0.01)
    sat_red_area_fraction = float(sat_red_mask.float().mean().item())

    # 2. CIE 1931 XYZ -> CIE 1976 UCS (u', v') 변환 및 색도 차이 Delta u'v' 계산
    x_cie = 0.4124564 * r_lin + 0.3575761 * g_lin + 0.1804375 * b_lin
    y_cie = 0.2126729 * r_lin + 0.7151522 * g_lin + 0.0721750 * b_lin
    z_cie = 0.0193339 * r_lin + 0.1191920 * g_lin + 0.9503041 * b_lin
    denom = x_cie + 15.0 * y_cie + 3.0 * z_cie
    valid_chroma = denom > 1e-4

    # D65 백색점 기본값 (u'_n = 0.1978, v'_n = 0.4683)
    u_prime = torch.where(valid_chroma, (4.0 * x_cie) / torch.clamp(denom, min=1e-6), torch.full_like(x_cie, 0.1978))
    v_prime = torch.where(valid_chroma, (9.0 * y_cie) / torch.clamp(denom, min=1e-6), torch.full_like(y_cie, 0.4683))

    non_red_mask = (~sat_red_mask) & valid_chroma
    if sat_red_mask.any() and non_red_mask.any():
        u_red_mean = u_prime[sat_red_mask].mean()
        v_red_mean = v_prime[sat_red_mask].mean()
        dist_map = torch.sqrt((u_prime[non_red_mask] - u_red_mean) ** 2 + (v_prime[non_red_mask] - v_red_mean) ** 2)
        delta_uv = float(torch.quantile(dist_map, 0.90).item())
    else:
        delta_uv = 0.0

    chroma_satisfied = delta_uv > 0.2
    is_danger = (sat_red_area_fraction > area_threshold) and chroma_satisfied
    verdict = "위험 (DANGER)" if is_danger else "안전 (SAFE)"

    # 방식 1, 2, 3 확률 산출
    red_gate = float(torch.clamp(red_ratio[sat_red_mask].mean() / 0.8 if sat_red_mask.any() else red_ratio.max() / 0.8, 0.0, 1.0).item())
    chroma_gate = min(1.0, delta_uv / 0.2)
    eta = (sat_red_area_fraction / area_threshold) * red_gate * chroma_gate
    prob_method_1 = (eta ** 2 / (1.0 + eta ** 2)) * 100.0
    prob_method_2 = (sat_red_area_fraction * 100.0) if chroma_satisfied else 0.0
    prob_method_3 = compute_cortical_risk_prob(sat_red_area_fraction, quality_gate=red_gate * chroma_gate)

    label_pct = int(round(area_threshold * 100))
    return pd.DataFrame([{
        "판정 결과": verdict,
        "판정 근거 확률 - 방식 1 (임계치 달성도 기반)": f"{prob_method_1:.2f}%",
        "판정 근거 확률 - 방식 2 (위험 조건 충족 면적 비율)": f"{prob_method_2:.2f}%",
        "판정 근거 확률 - 방식 3 (시피질 자극 Q 기반 환자 위험도)": f"{prob_method_3:.2f}%"
    }], index=[f"포화 적색 영역 (면적 기준 {label_pct}%)"])

df_cell_7 = evaluate_saturated_red(IMAGE_PATH, area_threshold=0.25)
display(df_cell_7)

,판정 결과,판정 근거 확률 - 방식 1 (임계치 달성도 기반),판정 근거 확률 - 방식 2 (위험 조건 충족 면적 비율),판정 근거 확률 - 방식 3 (시피질 자극 Q 기반 환자 위험도)
포화 적색 영역 (면적 기준 25%),안전 (SAFE),0.00%,0.00%,0.00%


### [코드 셀 8] 포화 적색 영역 판정 알고리즘 — 면적 임계치 40%

In [10]:

df_cell_8 = evaluate_saturated_red(IMAGE_PATH, area_threshold=0.40)
display(df_cell_8)

,판정 결과,판정 근거 확률 - 방식 1 (임계치 달성도 기반),판정 근거 확률 - 방식 2 (위험 조건 충족 면적 비율),판정 근거 확률 - 방식 3 (시피질 자극 Q 기반 환자 위험도)
포화 적색 영역 (면적 기준 40%),안전 (SAFE),0.00%,0.00%,0.00%
